# T02 · Hyperparameter search for an A2C network

The manual fine tuning showed which settings matter most (the entropy schedules) and roughly which values
work. This notebook searches them systematically with Optuna (TPE sampler) for one network. A trial trains
the agent through the three curriculum stages and is scored by its final score against the rule-based agent.
The search covers the learning rate of every stage and one entropy coefficient $c_H$, which is the starting
value of every stage's entropy schedule and decays to `ENTROPY_FINAL`.

Run the notebook once per network: `NETWORK_NAME = "FC128"` (Kartal) writes to `runs/a2c_hpo/` and
`NETWORK_NAME = "Conv1x1"` (Skynet) to `runs/a2c_hpo_1x1/`.

**Quality gate.** After each of the first two stages (`collect` and `crates`) the agent plays a few games on
that stage's board. If it collects less than half of the coins, the trial is stopped early (Optuna calls this
*pruning*). Stopped trials remain in the study and count as bad results.

The study is saved in `study.db`, so running the notebook again continues an interrupted search.

**Writes** `runs/a2c_hpo[_1x1]/`: `trials.csv` (one row per trial), `best.json`, `config.json` and `study.db`.
**Used by** A04 and T03 (which reads `best.json`).

In [ ]:
import os

os.environ.setdefault("JAX_COMPILATION_CACHE_DIR", os.path.expanduser("~/.cache/jax"))
os.environ.setdefault("CUDA_VISIBLE_DEVICES", "0")
os.environ.setdefault("XLA_PYTHON_CLIENT_MEM_FRACTION", "0.15")

import json
import pathlib
import time

import equinox as eqx
import jax
import jax.numpy as jnp
import numpy as np
import optuna
import pandas as pd

from bomberman.agents import A2CObjective, PolicyAgent, RuleBasedAgent
from bomberman.agents.networks import KartalCNNActorCriticNetwork, SkynetCNNActorCriticNetwork 
from bomberman.game import Game
from bomberman.training import (
    CoinPickupReward,
    DeathPenalty,
    KillReward,
    PBRSReward,
    Rollout,
    StepPenalty,
    Training,
    UrgentCrateReward,
    UrgentStepPenalty,
    coin_task_settings,
    crate_task_settings,
    versus_task_settings,
)
from bomberman.training.scenarios import NoReward

HERE = pathlib.Path.cwd().resolve()  # run from the notebook's folder or from the repository root
EXP = next(p / "experiments" for p in (HERE, *HERE.parents) if (p / "experiments" / "README.md").exists())
print(f"backend: {jax.default_backend()}")

## Configuration

In [ ]:
SIZE = 9
NETWORKS = {"FC128": KartalCNNActorCriticNetwork, "Conv1x1": SkynetCNNActorCriticNetwork}
RUN_NAMES = {"FC128": "a2c_hpo", "Conv1x1": "a2c_hpo_1x1"}
NETWORK_NAME = "FC128"  # the network to tune
NETWORK = NETWORKS[NETWORK_NAME]
OUT = EXP / "runs" / RUN_NAMES[NETWORK_NAME]
OUT.mkdir(parents=True, exist_ok=True)
print(f"tuning {NETWORK_NAME} -> {OUT}")
BUDGET = dict(trials=64, num_envs=150, iterations=(1000, 2000, 1000), eval_episodes=128, gate_episodes=64)
QUALITY_GATE = {"collect": 0.5, "crates": 0.5}  # share of coins the agent must collect to continue
LEARNING_RATE = (1e-4, 3e-3)  # search range, sampled on a log scale
ENTROPY_COEF = (5e-3, 1e-1)  # start value of c_H in every stage, sampled on a log scale
ENTROPY_FINAL = 5e-4  # final value of c_H in every stage

EVAL_SETTINGS = versus_task_settings(size=SIZE, crate_density=0.35, coins=9, max_steps=200, agents=2)

## Curriculum

One function per curriculum stage. Each one sets up the stage's board and reward, continues training the
agent it gets from the previous stage with a new entropy schedule, and returns `(agent, history, info)`.
The default values come from the manual fine tuning; the search replaces them.

In [ ]:
def train_stage(agent, key, settings, reward, *, iterations, num_envs, entropy, learning_rate=3e-4,
                forbid_bomb=False, opponents=(), checkpointer=None):
    c0, c1 = entropy  # c_H decays linearly from c0 to c1 during the stage
    agent = agent.configure(objective=agent.objective.configure(entropy_coef=c0, entropy_final=c1))
    trainer = Training(settings, [agent, *opponents], reward_fn=reward, forbid_bomb=forbid_bomb,
                       num_envs=num_envs, learning_rate=learning_rate, checkpointer=checkpointer)
    agent, *_ = trainer.fit(key, iterations)
    info = dict(iterations=iterations, entropy=entropy, learning_rate=learning_rate, forbid_bomb=forbid_bomb,
                settings=settings, reward=reward, opponents=[type(o).__name__ for o in opponents])
    return agent, trainer.history, info


def collect(agent, key, *, entropy=(0.02, 2e-3), **kwargs):
    """Stage 1, collect coins: no crates, 8 coins, no bombs allowed."""
    settings = coin_task_settings(size=SIZE, coins=8, max_steps=96)
    reward = CoinPickupReward() + StepPenalty(step=0.02)
    return train_stage(agent, key, settings, reward, entropy=entropy, forbid_bomb=True, **kwargs)


def crates(agent, key, *, entropy=(0.06, 0.01), **kwargs):
    """Stage 2, bomb crates: crate density 0.18, 6 coins, most of them under crates."""
    settings = crate_task_settings(size=SIZE, coins=6, crate_density=0.18, max_steps=140)
    reward = CoinPickupReward() + UrgentCrateReward() + DeathPenalty(death=2.0) + UrgentStepPenalty()
    return train_stage(agent, key, settings, reward, entropy=entropy, **kwargs)


def versus(agent, key, *, entropy=(0.04, 0.02), pbrs=True, **kwargs):
    """Stage 3, 1 vs. 1 against the rule-based agent, with potential-based reward shaping."""
    settings = versus_task_settings(size=SIZE, coins=6, crate_density=0.18, max_steps=140, agents=2)
    reward = (CoinPickupReward() + KillReward() + DeathPenalty(death=3.0) + StepPenalty(step=0.02)
              + UrgentCrateReward(urgent_crate_scale=1.0) + PBRSReward(settings, enabled=pbrs))
    return train_stage(agent, key, settings, reward, entropy=entropy, opponents=(RuleBasedAgent(),), **kwargs)


CURRICULUM = (collect, crates, versus)

## Evaluation

In [ ]:
EVAL_ROLLOUT = Rollout(Game(settings=EVAL_SETTINGS), NoReward(), EVAL_SETTINGS.rules.max_steps)


def seat_stats(out, seat=0):
    """The result of every game for one player. A self-kill is a death in the player's own explosion."""
    states = out.states
    after = jax.tree.map(lambda a: a[:, 1:], states)
    died = states.alive[:, :-1, seat] & ~states.alive[:, 1:, seat]
    hit = jax.vmap(jax.vmap(lambda board: board.explosions.covers(board.agent_positions)))(after.board)
    score = states.score[:, -1]
    rival = jnp.delete(score, seat, axis=1, assume_unique_indices=True).max(axis=1)
    return dict(
        score=score[:, seat], rival_score=rival, win=score[:, seat] > rival, coins=out.coins[:, seat],
        death=~out.alive[:, seat], self_kill=(died & hit[:, :, seat, seat]).any(axis=1), steps=out.steps,
    )


@eqx.filter_jit
def evaluate(network, key, episodes):
    agent = PolicyAgent(network, A2CObjective())
    return seat_stats(EVAL_ROLLOUT([agent, RuleBasedAgent()], key, greedy=True, num_envs=episodes))

## Quality gate

In [ ]:
GATE_ROLLOUTS = {}  # one rollout per stage board, so each is compiled only once


@eqx.filter_jit
def coins_collected(network, rollout, key, episodes):
    agent = PolicyAgent(network, A2CObjective())
    return rollout([agent], key, greedy=True, num_envs=episodes).coins[:, 0].mean()


def quality_gate(trial, stage, network, info, key):
    """Stop the trial (optuna.TrialPruned) if the agent collects too few coins on the stage's board."""
    if stage.__name__ not in QUALITY_GATE:
        return
    settings = info["settings"]
    rollout = GATE_ROLLOUTS.setdefault(stage.__name__, Rollout(
        Game(settings=settings), NoReward(), settings.rules.max_steps, forbid_bomb=info["forbid_bomb"]))
    share = float(coins_collected(network, rollout, key, BUDGET["gate_episodes"])) / settings.scenario.coin_count
    trial.set_user_attr(f"gate_{stage.__name__}", share)
    if share < QUALITY_GATE[stage.__name__]:
        trial.set_user_attr("stopped_after", stage.__name__)
        raise optuna.TrialPruned(f"{stage.__name__}: {share:.0%} of the coins collected")

## Objective

The sampled values are passed as Python floats. As JAX arrays, the optimiser would treat them as trainable weights.

In [ ]:
def objective(trial):
    learning_rate = {stage.__name__: trial.suggest_float(f"learning_rate_{stage.__name__}", *LEARNING_RATE, log=True)
                     for stage in CURRICULUM}
    entropy_coef = trial.suggest_float("entropy_coef", *ENTROPY_COEF, log=True)

    init_key, train_key, gate_key, eval_key = jax.random.split(jax.random.key(trial.number), 4)
    agent = PolicyAgent(NETWORK.for_settings(EVAL_SETTINGS, init_key),
                        A2CObjective(gamma=0.99, entropy_coef=entropy_coef, entropy_final=ENTROPY_FINAL,
                                     value_coef=0.5, n_steps=5))
    keys = jax.random.split(train_key, len(CURRICULUM))
    for stage, iterations, key in zip(CURRICULUM, BUDGET["iterations"], keys):
        agent, _, info = stage(agent, key, iterations=iterations, num_envs=BUDGET["num_envs"],
                               learning_rate=learning_rate[stage.__name__],
                               entropy=(entropy_coef, ENTROPY_FINAL))
        quality_gate(trial, stage, agent.network, info, gate_key)

    stats = {k: float(np.mean(v)) for k, v in evaluate(agent.network, eval_key, BUDGET["eval_episodes"]).items()}
    for name in ("win", "self_kill", "coins"):
        trial.set_user_attr(name, stats[name])
    return stats["score"]

## Search

In [ ]:
optuna.logging.set_verbosity(optuna.logging.WARNING)
study = optuna.create_study(
    study_name=RUN_NAMES[NETWORK_NAME], storage=f"sqlite:///{OUT / 'study.db'}", load_if_exists=True,
    direction="maximize", sampler=optuna.samplers.TPESampler(seed=0),
)
FINISHED = (optuna.trial.TrialState.COMPLETE, optuna.trial.TrialState.PRUNED)


def report(study, trial):
    result = (f"score {trial.value:.3f}" if trial.state == optuna.trial.TrialState.COMPLETE
              else f"stopped by the quality gate after {trial.user_attrs['stopped_after']}")
    print(f"trial {trial.number:3d}: {result}")


done = sum(t.state in FINISHED for t in study.trials)
start = time.time()
study.optimize(objective, n_trials=max(BUDGET["trials"] - done, 0), callbacks=[report])
counts = pd.Series([t.state.name for t in study.trials]).value_counts()
print(f"{dict(counts)} after {(time.time() - start) / 60:.1f} min")

In [ ]:
trials = study.trials_dataframe(attrs=("number", "value", "params", "user_attrs", "state"))
trials = (trials[trials.state.isin(["COMPLETE", "PRUNED"])]
          .rename(columns=lambda c: {"number": "trial", "value": "score"}.get(c, c.removeprefix("params_").removeprefix("user_attrs_"))))
trials.to_csv(OUT / "trials.csv", index=False)

complete = [t for t in study.trials if t.state == optuna.trial.TrialState.COMPLETE]
best = max(complete, key=lambda t: t.value, default=None)  # None if every trial was stopped early
(OUT / "best.json").write_text(json.dumps(
    dict(score=best.value, trial=best.number, **best.params) if best else None, indent=2))
(OUT / "config.json").write_text(json.dumps(dict(
    size=SIZE, network=NETWORK.__name__, budget=BUDGET, sampler="TPESampler(seed=0)", entropy_final=ENTROPY_FINAL,
    bounds=dict(learning_rate=LEARNING_RATE, entropy_coef=ENTROPY_COEF), quality_gate=QUALITY_GATE,
    stages=[stage.__name__ for stage in CURRICULUM],
), indent=2))
trials.sort_values("score", ascending=False).head()